# CIFAR-10 Single-Head ViT with Standard LayerNorm and MLP Softmax

This notebook keeps the same small CIFAR-10 ViT setup (single head, 3 Transformer blocks) and trains from scratch with **standard PyTorch LayerNorm** but the **learned MLP Softmax replacement**.

### Standard LayerNorm

Every Transformer normalization layer, including the final normalization layer, uses `nn.LayerNorm(dim)`.

### Learned Softmax replacement

For every attention-score row $s_1,\ldots,s_N$, a direct row-wise MLP is reused across all positions:

$$L_t=f_\theta(L_{t-1},s_t)$$

The row-wise predictor uses a **65 → 32 → 1** MLP by default. The final attention weight is

$$A_i=\exp(s_i-L_N).$$

There is no standard Softmax, no teacher Softmax, and no auxiliary loss. This notebook isolates the effect of the MLP Softmax replacement while keeping LayerNorm standard.


In this variant, the learned attention weights use $2^{s_i-L}$ instead of $e^{s_i-L}$. The exponent clamp remains `[-12, 0]` so that the only functional change from the E-base MLP version is the exponential base.

In [ ]:
# ============================================================
# Imports and configuration
# ============================================================

import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

DATA_ROOT = "./data"
OUT_DIR = "./runs/cifar10_singlehead_vit_standard_ln_mlp_softmax_base2"

EPOCHS = 400
BATCH_SIZE = 256
NUM_WORKERS = 4
LR = 1e-3
WEIGHT_DECAY = 0.05
SEED = 42

IMAGE_SIZE = 32
PATCH_SIZE = 4
DIM = 128
DEPTH = 3
MLP_RATIO = 4.0
DROPOUT = 0.1
ATTN_DROPOUT = 0.0

# MLP Softmax hidden width.
SOFTMAX_MLP_HIDDEN = 32


In [ ]:
# ============================================================
# Reproducibility
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1. Standard LayerNorm


In [ ]:
# ============================================================
# Standard LayerNorm
# ============================================================

# This ablation uses PyTorch's original LayerNorm directly:
#     nn.LayerNorm(dim)
#
# It is instantiated inside each Transformer block and as final_norm.


## 2. Tiny MLP Softmax replacement


In [ ]:
# ============================================================
# Direct MLP replacement for Softmax normalization (base 2)
# ============================================================

class MLPSoftmaxReplacement(nn.Module):
    """
    For each complete attention-score row s_1 ... s_N:

        L = MLP([s_1, ..., s_N])
        A_i = 2 ** (s_i - L)

    All scores in a row are available simultaneously, so no MLP
    scan is used. Each query row is processed independently and all
    rows can be evaluated in parallel.
    """

    def __init__(self, num_tokens, hidden_dim=32):
        super().__init__()

        self.num_tokens = num_tokens

        self.normalizer_mlp = nn.Sequential(
            nn.Linear(num_tokens, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

        # Keep the final predicted normalization state close to the
        # uniform-attention scale at initialization.
        first_layer = self.normalizer_mlp[0]
        final_layer = self.normalizer_mlp[-1]

        nn.init.trunc_normal_(first_layer.weight, std=0.02)
        nn.init.zeros_(first_layer.bias)

        nn.init.normal_(final_layer.weight, mean=0.0, std=1e-3)
        # If initial scores are near zero, L ~= log2(N) gives
        # 2**(score - L) ~= 1/N, close to a uniform attention scale.
        nn.init.constant_(final_layer.bias, math.log2(num_tokens))

    def forward(self, scores):
        # scores: [B, N_query, N_key]
        B, Nq, Nk = scores.shape

        if Nk != self.num_tokens:
            raise ValueError(
                f"MLPSoftmaxReplacement expects N_key={self.num_tokens}, "
                f"but received N_key={Nk}."
            )

        # Every complete query row is fed directly to one MLP.
        rows = scores.reshape(B * Nq, Nk)

        # One learned scalar normalization state per query row.
        L = self.normalizer_mlp(rows)
        L = L.reshape(B, Nq, 1)

        # No analytical Softmax sum or division is used.
        exponent = torch.clamp(
            scores - L,
            min=-12.0,
            max=0.0,
        )

        attn = torch.exp2(exponent)

        return attn


## 3. Single-head attention

In [ ]:
# ============================================================
# Single-head self-attention
# ============================================================

class SingleHeadSelfAttention(nn.Module):

    def __init__(
        self,
        dim,
        num_tokens,
        softmax_hidden=32,
        attn_dropout=0.0,
        proj_dropout=0.0,
    ):
        super().__init__()

        self.dim = dim
        self.scale = dim ** -0.5

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

        self.learned_softmax = MLPSoftmaxReplacement(
            num_tokens=num_tokens,
            hidden_dim=softmax_hidden,
        )

        self.attn_dropout = nn.Dropout(attn_dropout)
        self.proj_dropout = nn.Dropout(proj_dropout)

    def forward(self, x):
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        scores = torch.matmul(
            q,
            k.transpose(-2, -1),
        ) * self.scale

        # No torch.softmax here.
        attn = self.learned_softmax(scores)
        attn = self.attn_dropout(attn)

        out = torch.matmul(attn, v)
        out = self.out_proj(out)
        out = self.proj_dropout(out)

        return out


## 4. MLP and Transformer block

In [ ]:
# ============================================================
# MLP
# ============================================================

class MLP(nn.Module):

    def __init__(self, dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x


# ============================================================
# Transformer block
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(
        self,
        dim,
        num_tokens,
        mlp_ratio=4.0,
        dropout=0.0,
        attn_dropout=0.0,
        softmax_hidden=8,
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)

        self.attn = SingleHeadSelfAttention(
            dim=dim,
            num_tokens=num_tokens,
            softmax_hidden=softmax_hidden,
            attn_dropout=attn_dropout,
            proj_dropout=dropout,
        )

        self.norm2 = nn.LayerNorm(dim)

        self.mlp = MLP(
            dim=dim,
            hidden_dim=int(dim * mlp_ratio),
            dropout=dropout,
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


## 5. CIFAR-10 ViT

In [ ]:
# ============================================================
# CIFAR-10 single-head ViT
# ============================================================

class CIFAR10SingleHeadViT(nn.Module):

    def __init__(
        self,
        image_size=32,
        patch_size=4,
        in_channels=3,
        num_classes=10,
        dim=128,
        depth=3,
        mlp_ratio=4.0,
        dropout=0.1,
        attn_dropout=0.0,
        softmax_hidden=8,
    ):
        super().__init__()

        assert image_size % patch_size == 0

        self.image_size = image_size
        self.patch_size = patch_size
        self.dim = dim

        patches_per_side = image_size // patch_size
        self.num_patches = patches_per_side ** 2
        self.num_tokens = self.num_patches + 1

        patch_dim = in_channels * patch_size * patch_size

        self.patch_proj = nn.Linear(patch_dim, dim)

        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.num_tokens, dim)
        )
        self.pos_drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(
                dim=dim,
                num_tokens=self.num_tokens,
                mlp_ratio=mlp_ratio,
                dropout=dropout,
                attn_dropout=attn_dropout,
                softmax_hidden=softmax_hidden,
            )
            for _ in range(depth)
        ])

        # Standard final LayerNorm.
        self.final_norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # Initialize ordinary Linear layers as in the previous baseline,
        # while preserving the special initialization of the MLP
        # Softmax cell output layer.
        for name, module in self.named_modules():
            if not isinstance(module, nn.Linear):
                continue

            if "normalizer_mlp.2" in name:
                continue

            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def patchify(self, x):
        p = self.patch_size

        x = x.unfold(2, p, p).unfold(3, p, p)
        x = x.permute(0, 2, 3, 1, 4, 5).contiguous()
        x = x.view(x.size(0), self.num_patches, -1)
        return x

    def forward(self, x):
        x = self.patchify(x)
        x = self.patch_proj(x)

        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1)

        x = x + self.pos_embed
        x = self.pos_drop(x)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        cls_out = x[:, 0]
        logits = self.head(cls_out)
        return logits


def build_model():
    return CIFAR10SingleHeadViT(
        image_size=IMAGE_SIZE,
        patch_size=PATCH_SIZE,
        num_classes=10,
        dim=DIM,
        depth=DEPTH,
        mlp_ratio=MLP_RATIO,
        dropout=DROPOUT,
        attn_dropout=ATTN_DROPOUT,
        softmax_hidden=SOFTMAX_MLP_HIDDEN,
    )


## 6. CIFAR-10 data loaders

In [ ]:
# ============================================================
# Dataset
# ============================================================

def make_loaders(
    data_root="./data",
    batch_size=256,
    num_workers=4,
    seed=42,
):

    train_tf = transforms.Compose([
        transforms.RandomCrop(
            32,
            padding=4,
        ),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(
                0.4914,
                0.4822,
                0.4465,
            ),
            std=(
                0.2470,
                0.2435,
                0.2616,
            ),
        ),
    ])

    eval_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(
                0.4914,
                0.4822,
                0.4465,
            ),
            std=(
                0.2470,
                0.2435,
                0.2616,
            ),
        ),
    ])

    full_train_aug = datasets.CIFAR10(
        root=data_root,
        train=True,
        download=True,
        transform=train_tf,
    )

    full_train_eval = datasets.CIFAR10(
        root=data_root,
        train=True,
        download=False,
        transform=eval_tf,
    )

    test_set = datasets.CIFAR10(
        root=data_root,
        train=False,
        download=True,
        transform=eval_tf,
    )

    g = torch.Generator().manual_seed(seed)

    indices = torch.randperm(
        len(full_train_aug),
        generator=g,
    ).tolist()

    train_indices = indices[:45_000]
    val_indices = indices[45_000:]

    train_set = Subset(
        full_train_aug,
        train_indices,
    )

    # Same training samples, but augmentation disabled.
    train_eval_set = Subset(
        full_train_eval,
        train_indices,
    )

    val_set = Subset(
        full_train_eval,
        val_indices,
    )

    loader_kwargs = dict(
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
    )

    train_loader = DataLoader(
        train_set,
        shuffle=True,
        drop_last=True,
        **loader_kwargs,
    )

    train_eval_loader = DataLoader(
        train_eval_set,
        shuffle=False,
        drop_last=False,
        **loader_kwargs,
    )

    val_loader = DataLoader(
        val_set,
        shuffle=False,
        drop_last=False,
        **loader_kwargs,
    )

    test_loader = DataLoader(
        test_set,
        shuffle=False,
        drop_last=False,
        **loader_kwargs,
    )

    return (
        train_loader,
        train_eval_loader,
        val_loader,
        test_loader,
    )


train_loader, train_eval_loader, val_loader, test_loader = make_loaders(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seed=SEED,
)

print(
    len(train_loader.dataset),
    len(val_loader.dataset),
    len(test_loader.dataset),
)


### Numerical stabilization used in this version

This ablation keeps **standard LayerNorm** and only replaces Softmax:

- LayerNorm: standard `nn.LayerNorm`
- learned Softmax: $L_t=f_\theta(L_{t-1},s_t)$, followed by $\exp(s_i-L_N)$

Two safeguards are retained for the learned attention path:

1. The exponent $s_i-L_N$ is clipped to $[-12,0]$, preventing `exp()` overflow.
2. Gradient norm is clipped to 1.0.

There is no standard Softmax, no Softmax teacher, and no auxiliary normalization loss.


## 7. Training and evaluation

In [ ]:
# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader,
    device,
):
    model.eval()

    criterion = nn.CrossEntropyLoss()

    total = 0
    correct = 0
    loss_sum = 0.0

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True,
        )

        labels = labels.to(
            device,
            non_blocking=True,
        )

        logits = model(images)
        loss = criterion(
            logits,
            labels,
        )

        total += labels.size(0)

        correct += (
            logits.argmax(dim=1)
            == labels
        ).sum().item()

        loss_sum += (
            loss.item()
            * labels.size(0)
        )

    return (
        loss_sum / total,
        correct / total,
    )


# ============================================================
# Training
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=100,
    lr=3e-4,
    weight_decay=0.05,
    out_dir="./runs/cifar10_singlehead_vit_standard_ln_mlp_softmax_base2",
):

    out_dir = Path(out_dir)
    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=epochs,
        )
    )

    best_val_acc = 0.0

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        total = 0
        correct = 0
        loss_sum = 0.0

        for images, labels in train_loader:

            images = images.to(
                device,
                non_blocking=True,
            )

            labels = labels.to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(images)

            loss = criterion(
                logits,
                labels,
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    "Non-finite loss detected. "
                    "The learned MLP attention path is numerically unstable."
                )

            loss.backward()

            # Keep the learned normalization state numerically stable
            # during early training.
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            total += labels.size(0)

            correct += (
                logits.argmax(dim=1)
                == labels
            ).sum().item()

            loss_sum += (
                loss.item()
                * labels.size(0)
            )

        scheduler.step()

        train_loss = loss_sum / total
        train_acc = correct / total

        val_loss, val_acc = evaluate(
            model,
            val_loader,
            device,
        )

        print(
            f"Epoch {epoch:03d}/{epochs} | "
            f"lr={scheduler.get_last_lr()[0]:.3e} | "
            f"train loss={train_loss:.4f} "
            f"acc={train_acc:.4f} | "
            f"val loss={val_loss:.4f} "
            f"acc={val_acc:.4f}"
        )

        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "val_acc": val_acc,
            },
            out_dir / "last.pt",
        )

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            torch.save(
                {
                    "epoch": epoch,
                    "model": model.state_dict(),
                    "val_acc": val_acc,
                },
                out_dir / "best.pt",
            )

    print(
        f"Best validation accuracy: "
        f"{best_val_acc:.4f}"
    )


def load_checkpoint(
    model,
    ckpt_path,
    device,
):

    ckpt = torch.load(
        ckpt_path,
        map_location=device,
    )

    model.load_state_dict(
        ckpt["model"]
    )

    print(
        f"Loaded checkpoint: {ckpt_path} | "
        f"epoch={ckpt.get('epoch', '?')} | "
        f"val_acc={ckpt.get('val_acc', '?')}"
    )


## 8. Build and train from scratch

In [ ]:
# ============================================================
# Build model
# ============================================================

model = build_model().to(device)

num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(model)
print(f"Parameters: {num_params:,}")


In [ ]:
# ============================================================
# Train from scratch
# ============================================================

train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    out_dir=OUT_DIR,
)


In [ ]:
# ============================================================
# Load best checkpoint and evaluate
# ============================================================

best_ckpt = Path(OUT_DIR) / "best.pt"
load_checkpoint(model, best_ckpt, device)

val_loss, val_acc = evaluate(model, val_loader, device)
test_loss, test_acc = evaluate(model, test_loader, device)

print(f"Validation: loss={val_loss:.4f}, acc={val_acc:.4f}")
print(f"Test:       loss={test_loss:.4f}, acc={test_acc:.4f}")


## Notes

This notebook is the **standard LayerNorm + learned MLP Softmax** ablation. It intentionally uses no Softmax teacher, auxiliary regression loss, explicit normalization loss, or quantization. The only learned replacement being tested is the MLP Softmax mechanism.
